In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings("ignore")

In [7]:
APPLIANCE_COLS = [
    "elec_ceiling_fan_kwh",
    "elec_clothes_washer_kwh",
    "elec_cooling_kwh",
    "elec_freezer_kwh",
    "elec_heating_kwh",
    "elec_hot_water_kwh",
    "elec_lighting_exterior_kwh",
    "elec_lighting_interior_kwh",
    "elec_plug_loads_kwh",
    "elec_range_oven_kwh",
    "elec_refrigerator_kwh",
    "elec_television_kwh",
]

CATEGORY_TO_COL = {
    "cooling":      "elec_cooling_kwh",
    "heating":      "elec_heating_kwh",
    "water_heater": "elec_hot_water_kwh",
    "washer":       "elec_clothes_washer_kwh",
    "refrigerator": "elec_refrigerator_kwh",
    "freezer":      "elec_freezer_kwh",
    "lighting":     "elec_lighting_interior_kwh",
    "ext_lighting": "elec_lighting_exterior_kwh",
    "plug_loads":   "elec_plug_loads_kwh",
    "tv":           "elec_television_kwh",
    "ceiling_fan":  "elec_ceiling_fan_kwh",
    "range_oven":   "elec_range_oven_kwh",
}

CATEGORY_CONSTRAINTS = {
    # max_daily_runtime: realistic upper bounds for household appliances
    # max_hours_without_running: how long the appliance can be skipped
    "washer":       {"max_daily_runtime": 1,  "max_hours_without_running": 24},
    "water_heater": {"max_daily_runtime": 2,  "max_hours_without_running": 12},  # was 6 — too high
    "range_oven":   {"max_daily_runtime": 2,  "max_hours_without_running": 24},
    "cooling":      {"max_daily_runtime": 10, "max_hours_without_running": 24},  # was 16
    "heating":      {"max_daily_runtime": 10, "max_hours_without_running": 24},  # was 16
}

MAX_DEFERRABLE = 12  # max individual deferrable appliances any household can have

class ApplianceRegistry:
    def __init__(self, appliance_list):
        self.appliances = appliance_list
        self.fixed      = [a for a in appliance_list if not a.get("deferrable", False)]
        self.deferrable = [a for a in appliance_list if a.get("deferrable", False)]

    @property
    def n_deferrable(self):
        return len(self.deferrable)

    def get_category_col(self, appliance):
        return CATEGORY_TO_COL.get(appliance["name"])

    def get_watt_share(self, appliance):
        same_category = [a for a in self.appliances if a["name"] == appliance["name"]]
        total_watts = sum(a["watts"] for a in same_category)
        return appliance["watts"] / total_watts if total_watts > 0 else 0.0

    def get_fixed_cols(self):
        return list({
            CATEGORY_TO_COL[a["name"]]
            for a in self.fixed
            if a["name"] in CATEGORY_TO_COL
        })

    def get_max_daily_runtime(self, appliance):
        """Look up max daily runtime from category constraints."""
        return CATEGORY_CONSTRAINTS.get(
            appliance["name"], {}
        ).get("max_daily_runtime", 24)

    def get_max_hours_without_running(self, appliance):
        """Look up max hours without running from category constraints."""
        return CATEGORY_CONSTRAINTS.get(
            appliance["name"], {}
        ).get("max_hours_without_running", 24)

print("Constraints loaded:")
for name, constraints in CATEGORY_CONSTRAINTS.items():
    print(f"  {name}: {constraints}")

# And verify ApplianceRegistry reads them
test_registry = ApplianceRegistry([
    {"name": "washer", "label": "Washer", "watts": 1200, "deferrable": True},
    {"name": "water_heater", "label": "WH", "watts": 2400, "deferrable": True},
    {"name": "range_oven",   "label": "Oven", "watts": 1500, "deferrable": True},
])
for a in test_registry.deferrable:
    print(f"{a['name']}: max_runtime={test_registry.get_max_daily_runtime(a)}, "
          f"max_hours={test_registry.get_max_hours_without_running(a)}")

Constraints loaded:
  washer: {'max_daily_runtime': 1, 'max_hours_without_running': 24}
  water_heater: {'max_daily_runtime': 2, 'max_hours_without_running': 12}
  range_oven: {'max_daily_runtime': 2, 'max_hours_without_running': 24}
  cooling: {'max_daily_runtime': 10, 'max_hours_without_running': 24}
  heating: {'max_daily_runtime': 10, 'max_hours_without_running': 24}
washer: max_runtime=1, max_hours=24
water_heater: max_runtime=2, max_hours=12
range_oven: max_runtime=2, max_hours=24


In [8]:
# Cell — NILM model definition (needed to load saved models)
class Seq2Point(torch.nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(window_size, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(64, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

# Cell — NILM inference functions
def load_seq2point_models(model_dir="nilm_models", window_size=11):
    scaler = joblib.load(os.path.join(model_dir, "nilm_scaler.pkl"))
    models = {}
    for col in APPLIANCE_COLS:
        path = os.path.join(model_dir, f"nilm_{col}.pt")
        if not os.path.exists(path):
            print(f"Warning: model not found for {col}, skipping")
            continue
        model = Seq2Point(window_size)
        model.load_state_dict(torch.load(path, map_location="cpu"))
        model.eval()
        models[col] = model
    print(f"Loaded {len(models)} appliance models")
    return models, scaler


def disaggregate_seq2point(ups_csv_path, models, scaler, window_size=11):
    """
    FIX #4: NILM disaggregates PAST aggregate consumption into per-appliance loads.
    To produce a NEXT-WEEK forecast, we use the disaggregated past week as a
    naive seasonal-persistence forecast (assume next week ~= last week pattern).
    The HVAC components are then re-scaled by the actual weather forecast in
    scale_nilm_by_weather().
    """
    ups = pd.read_csv(ups_csv_path, parse_dates=["timestamp"])
    ups = ups.sort_values("timestamp").reset_index(drop=True)

    if len(ups) < 168:
        raise ValueError(f"UPS data has {len(ups)} rows — need at least 168")

    ups = ups.tail(168).reset_index(drop=True)
    aggregate      = ups["total_kwh"].values.astype(np.float32)
    aggregate_norm = scaler.transform(
        aggregate.reshape(-1, 1)
    ).flatten().astype(np.float32)

    half   = window_size // 2
    padded = np.pad(aggregate_norm, (half, half), mode="edge")
    windows = np.array([padded[i : i + window_size] for i in range(len(aggregate))])
    X = torch.FloatTensor(windows)

    predictions = {}
    with torch.no_grad():
        for col, model in models.items():
            pred = model(X).numpy().flatten()
            predictions[col] = np.clip(pred, 0, None)

    for col in APPLIANCE_COLS:
        if col not in predictions:
            predictions[col] = np.zeros(168)

    # Align disaggregated past week to start at hour-of-week 0 of the FORECAST window.
    # The last hour of UPS history corresponds to "now"; next forecast hour is "now+1".
    # We roll so that the historical hour-of-week pattern aligns with the upcoming week.
    last_ts = ups["timestamp"].iloc[-1]
    # Hour-of-week the forecast STARTS (one hour after last historical sample)
    forecast_start_hour_of_week = ((last_ts.dayofweek * 24 + last_ts.hour) + 1) % 168
    # The disaggregation is indexed 0..167 in chronological order of the past week.
    # Its hour-of-week 0 corresponds to: (last_ts hour-of-week - 167) % 168.
    hist_first_how = (last_ts.dayofweek * 24 + last_ts.hour - 167) % 168
    # Roll so position 0 of the output corresponds to forecast_start_hour_of_week.
    shift = (forecast_start_hour_of_week - hist_first_how) % 168

    df_out = pd.DataFrame(predictions).reset_index(drop=True)
    df_out = pd.DataFrame(
        {col: np.roll(df_out[col].values, -shift) for col in df_out.columns}
    )
    return df_out

# Cell — load models once at startup (gracefully handle missing model dir)
try:
    seq2point_models, nilm_scaler = load_seq2point_models()
except FileNotFoundError as e:
    print(f"NILM models unavailable ({e}). NILM path will be disabled.")
    seq2point_models, nilm_scaler = {}, None

Loaded 12 appliance models


In [9]:
import pvlib
import pandas as pd
import numpy as np
import requests

def get_solar_from_data(df_week, panel_capacity_kw=5.0, latitude=36.7, longitude=-119.7,
                        tilt=None, azimuth=None):
    """
    FIX #5: Build proper monotonically-increasing timestamps for the 168-hour window
    instead of fake `day = 1 + day_of_week` which gave physically wrong solar geometry.
    We anchor to a known Monday and use the actual hour index 0..167.
    """
    # Anchor: pick a Monday in the same month as the data so the sun angle is
    # roughly correct for that season. Default to first Monday of the month.
    month = int(df_week["month"].iloc[0])
    year  = 2018  # any non-leap reference year
    # Find the first Monday of (year, month)
    first = pd.Timestamp(year=year, month=month, day=1)
    # weekday(): Monday=0 ... Sunday=6
    days_to_monday = (7 - first.weekday()) % 7
    anchor = first + pd.Timedelta(days=days_to_monday)
    # Align anchor's weekday with df_week's first row's day_of_week
    first_dow = int(df_week["day_of_week"].iloc[0])
    first_hour = int(df_week["hour"].iloc[0])
    anchor = anchor + pd.Timedelta(days=first_dow, hours=first_hour)

    timestamps = pd.date_range(start=anchor, periods=len(df_week), freq="h")

    weather = pd.DataFrame({
        "dni":        df_week["weather_direct_normal_radiation_w_m2"].values,
        "dhi":        df_week["weather_diffuse_horizontal_radiation_w_m2"].values,
        "ghi":        df_week["weather_global_horizontal_radiation_w_m2"].values,
        "temp_air":   df_week["weather_drybulb_temp_c"].values,
        "wind_speed": df_week["weather_wind_speed_m_s"].values,
    }, index=timestamps)

    return _run_pvlib(weather, latitude, longitude, panel_capacity_kw,
                      tilt=tilt, azimuth=azimuth)


def get_solar_and_features_from_forecast(latitude, longitude, start_date,
                                          panel_capacity_kw=5.0,
                                          tilt=None, azimuth=180):
    """
    Fetches forecast data and returns:
    1. solar_power: np.array (168,) for the RL environment
    2. weather_df: DataFrame with ALL columns matching your parquet for XGBoost
    """
    end_date = (pd.Timestamp(start_date) + pd.Timedelta(days=6)).strftime("%Y-%m-%d")

    response = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "windspeed_10m",
            "winddirection_10m",
            "shortwave_radiation",
            "direct_normal_irradiance",
            "diffuse_radiation",
        ],
        "timezone": "auto",
        "timeformat": "unixtime",
    }, timeout=10)

    data = response.json()

    if "hourly" not in data:
        raise ValueError(f"API error: {data.get('reason', 'check date range or connection')}")

    hourly = data["hourly"]
    times = pd.to_datetime(hourly["time"], unit="s", utc=True).tz_convert(data["timezone"])

    weather_df = pd.DataFrame({
        "weather_drybulb_temp_c":                    hourly["temperature_2m"],
        "weather_relative_humidity_pct":              hourly["relative_humidity_2m"],
        "weather_wind_speed_m_s":                     hourly["windspeed_10m"],
        "weather_wind_direction_deg":                 hourly["winddirection_10m"],
        "weather_global_horizontal_radiation_w_m2":   hourly["shortwave_radiation"],
        "weather_direct_normal_radiation_w_m2":       hourly["direct_normal_irradiance"],
        "weather_diffuse_horizontal_radiation_w_m2":  hourly["diffuse_radiation"],
        "hour":        times.hour,
        "month":       times.month,
        "day_of_week": times.dayofweek,
        "is_weekend":  times.dayofweek >= 5,
    }, index=times)

    pv_weather = weather_df.rename(columns={
        "weather_drybulb_temp_c":                   "temp_air",
        "weather_wind_speed_m_s":                   "wind_speed",
        "weather_direct_normal_radiation_w_m2":     "dni",
        "weather_diffuse_horizontal_radiation_w_m2":"dhi",
        "weather_global_horizontal_radiation_w_m2": "ghi",
    })
    solar_power = _run_pvlib(pv_weather, latitude, longitude, panel_capacity_kw, tilt, azimuth)

    temp_forecast = pd.to_numeric(
        weather_df["weather_drybulb_temp_c"], errors="coerce"
    ).fillna(20).values

    return solar_power, temp_forecast, weather_df


def _run_pvlib(weather, latitude, longitude, panel_capacity_kw, tilt=None, azimuth=None):
    """Shared PVLib logic used by both functions above.

    FIX #19: respect caller-provided azimuth (don't unconditionally overwrite).
    FIX #20: docstring placement.
    """
    tilt    = abs(latitude) if tilt is None else tilt
    # Default azimuth: south in N hemisphere, north in S hemisphere
    azimuth = (180 if latitude >= 0 else 0) if azimuth is None else azimuth

    location = pvlib.location.Location(latitude=latitude, longitude=longitude)

    weather = weather.copy()
    for col in ["dni", "dhi", "ghi", "temp_air", "wind_speed"]:
        weather[col] = pd.to_numeric(weather[col], errors="coerce").fillna(0).astype(float)

    solar_position = location.get_solarposition(weather.index)

    poa = pvlib.irradiance.get_total_irradiance(
        surface_tilt=tilt,
        surface_azimuth=azimuth,
        solar_zenith=solar_position["apparent_zenith"],
        solar_azimuth=solar_position["azimuth"],
        dni=weather["dni"],
        ghi=weather["ghi"],
        dhi=weather["dhi"],
    )

    temp_params = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS["sapm"]["open_rack_glass_glass"]
    cell_temp = pvlib.temperature.sapm_cell(
        poa_global=poa["poa_global"],
        temp_air=weather["temp_air"],
        wind_speed=weather["wind_speed"],
        **temp_params,
    )

    dc_power = pvlib.pvsystem.pvwatts_dc(
        effective_irradiance=poa["poa_global"],
        temp_cell=cell_temp,
        pdc0=panel_capacity_kw * 1000,
        gamma_pdc=-0.004,
    ) / 1000

    return dc_power.fillna(0).clip(lower=0).values  # shape (168,)


def profile_to_xgb_features(profile, weather_df):
    """
    Converts household profile answers into XGBoost-compatible features.
    weather_df: from get_solar_and_features_from_forecast (168 rows)
    """
    features = weather_df.copy()

    n_people = profile.get("n_people", 3)

    sqft_map = {
        ("apartment", 1): 600,  ("apartment", 2): 800,
        ("apartment", 3): 1000, ("apartment", 4): 1200,
        ("apartment", 5): 1400, ("villa", 1): 1000,
        ("villa", 2): 1400,     ("villa", 3): 1800,
        ("villa", 4): 2200,     ("villa", 5): 2600,
    }
    home_type = profile.get("home_type", "apartment")
    sqft = sqft_map.get((home_type, min(n_people, 5)), 1000)

    bedrooms = max(1, n_people // 2)

    hvac_map = {
        "split_ac":   "Room AC",
        "central_ac": "Central AC",
        "none":       "None",
    }
    hvac_type = hvac_map.get(profile.get("cooling_type", "split_ac"), "Room AC")

    features["sqft"]              = sqft
    features["bedrooms"]          = bedrooms
    features["hvac_cooling_type"] = hvac_type

    features = pd.get_dummies(features, columns=["bedrooms", "hvac_cooling_type"])

    return features, sqft, bedrooms, hvac_type


def scale_predictions_by_profile(appliance_df, profile, month):
    """
    FIX #22: do NOT zero out heating in summer / cooling in winter.
    The env handles the AC-vs-heating mutual-exclusion logic; we keep small
    seasonal scaling so the load forecast is realistic but non-zero.
    """
    df = appliance_df.copy().reset_index(drop=True)

    wake = profile.get("wake_time", 7)
    home = profile.get("home_during_day", False)

    # Scale hot water based on usage level
    hot_water_scale = {"low": 0.5, "medium": 1.0, "high": 1.5}
    df["elec_hot_water_kwh"] *= hot_water_scale.get(
        profile.get("hot_water_usage", "medium"), 1.0
    )

    # Scale cooling/heating based on home_during_day
    if not home:
        for hour in range(wake + 2, 17):
            df.loc[df.index % 24 == hour, "elec_cooling_kwh"] *= 0.3
            df.loc[df.index % 24 == hour, "elec_heating_kwh"] *= 0.3

    # Zero out washer on non-scheduled days
    washer_freq = profile.get("washer_frequency", "daily")
    washer_days = {
        "daily":        list(range(7)),
        "every_2_days": [0, 2, 4, 6],
        "weekly":       [0],
    }.get(washer_freq, list(range(7)))

    for day in range(7):
        if day not in washer_days:
            df.loc[day*24:(day+1)*24-1, "elec_clothes_washer_kwh"] = 0.0

    # Shift washer load to preferred time
    washer_time = profile.get("washer_preferred_time", "morning")
    washer_peak = {"morning": 9, "afternoon": 13, "evening": 19}.get(washer_time, 9)

    for day in range(7):
        if day in washer_days:
            day_total = df.loc[day*24:(day+1)*24-1, "elec_clothes_washer_kwh"].sum()
            df.loc[day*24:(day+1)*24-1, "elec_clothes_washer_kwh"] = 0.0
            for offset in range(2):
                hour_idx = day * 24 + washer_peak + offset
                if hour_idx < (day + 1) * 24:
                    df.loc[hour_idx, "elec_clothes_washer_kwh"] = day_total / 2

    # Scale plug loads by number of people
    n_people = profile.get("n_people", 3)
    df["elec_plug_loads_kwh"] *= (n_people / 3.0)

    # Soft seasonal scaling — let the env decide which one runs
    is_winter = month in [11, 12, 1, 2, 3]
    if is_winter:
        df["elec_cooling_kwh"] *= 0.05  # near-zero but not zero
    else:
        df["elec_heating_kwh"] *= 0.05

    return df.clip(lower=0)


def scale_nilm_by_weather(appliance_df, weather_df, month):
    """
    FIX #22: Same softening as above — don't fully zero out the off-season HVAC.
    """
    df = appliance_df.copy().reset_index(drop=True)

    temps = pd.to_numeric(
        weather_df["weather_drybulb_temp_c"], errors="coerce"
    ).fillna(20).values

    is_winter = month in [11, 12, 1, 2, 3]

    for i, temp in enumerate(temps):
        if not is_winter:
            cooling_scale = max(0, 1 + (temp - 25) * 0.08)
            df.loc[i, "elec_cooling_kwh"] *= cooling_scale
            df.loc[i, "elec_heating_kwh"] *= 0.05
        else:
            heating_scale = max(0, 1 + (18 - temp) * 0.10)
            df.loc[i, "elec_heating_kwh"] *= heating_scale
            df.loc[i, "elec_cooling_kwh"] *= 0.05

    return df.clip(lower=0)

In [10]:
import gymnasium as gym
import numpy as np

class SolarSchedulingEnv(gym.Env):
    """
    Major fixes applied:
      #2  deferrable_actual now uses each appliance's REAL hourly draw (watts/1000),
          not the predictor's category-share envelope, when it is run.
      #3  force_run threshold replaced with a per-appliance "max hours without running"
          + accumulated-skips counter (no more 1.0 kWh hardcode).
      #6  cooling/heating/oven are *deferrable* both in training and inference now
          (handled at the appliance_list level — env stays generic).
      #7  Pending normalization now uses an absolute reference (10 kWh) instead of
          the random battery capacity, so the policy generalizes across system sizes.
      #11 Reward weights rebalanced — unmet demand dominates, low-battery is soft.
      #12 min_soc enforced as a HARD constraint on discharge.
      #13 mutual_exclusion penalty applied BEFORE override (penalty informs the
          policy gradient before we mask the action).
      #16 Action thresholding gets a small deadband (>0.1) instead of >0.
      #17 reset() now resamples the entire scenario (building/system/appliances)
          when an episode_factory is provided.
      #25 Final ReLU + np.clip is fine; no change needed beyond doc.
    """
    def __init__(self, solar_forecast, temp_forecast, appliance_data, system_config, registry,
                 episode_factory=None):
        super().__init__()

        self.episode_factory = episode_factory  # callable -> (solar, temp, appliance_df, sys_cfg, registry)

        self._install_scenario(solar_forecast, temp_forecast, appliance_data, system_config, registry)

        # Action space: [charge, def_1, ..., def_MAX]
        # action[0]   in [-1, 1] -> charge command (mapped to [0, max_charge_rate])
        # action[1+i] in [-1, 1] -> deferrable i (run if > deadband)
        self.action_space = gym.spaces.Box(
            low=-1, high=1,
            shape=(1 + MAX_DEFERRABLE,),
            dtype=np.float32
        )

        # Observation space
        # 6 system + 24 solar + 24 fixed + 24 temp + (24*MAX) deferrable + MAX pending
        n_obs = 6 + 24 + 24 + 24 + (24 * MAX_DEFERRABLE) + MAX_DEFERRABLE
        self.observation_space = gym.spaces.Box(
            low=0, high=1, shape=(n_obs,), dtype=np.float32
        )

        self._reset_state()

    def _install_scenario(self, solar_forecast, temp_forecast, appliance_data, system_config, registry):
        """Set up everything that depends on the current scenario."""
        self.battery_capacity   = float(system_config["battery_capacity_kwh"])
        self.max_charge_rate    = float(system_config["max_charge_rate_kw"])
        self.max_discharge_rate = float(system_config["max_discharge_rate_kw"])
        self.min_soc            = float(system_config["min_soc_pct"])
        self.panel_capacity     = float(system_config["panel_capacity_kw"])
        self.registry           = registry
        self.temp_forecast      = np.asarray(temp_forecast, dtype=np.float32)
        self.solar_forecast     = np.asarray(solar_forecast, dtype=np.float32)

        appliance_data = appliance_data.copy()
        for col in APPLIANCE_COLS:
            if col not in appliance_data.columns:
                appliance_data[col] = 0.0

        fixed_cols = registry.get_fixed_cols()
        if fixed_cols:
            self.fixed_loads = appliance_data[fixed_cols].values.astype(np.float32)  # (168, n_fixed)
        else:
            self.fixed_loads = np.zeros((168, 0), dtype=np.float32)

        # FIX #2: store BOTH the predictor envelope (for obs) AND each appliance's
        # real hourly draw (kWh = watts/1000) for accounting when run.
        self.deferrable_envelope = []   # predictor share, used as obs hint
        self.deferrable_kwh_per_hour = []  # appliance true hourly draw in kWh
        for appliance in registry.deferrable:
            col   = registry.get_category_col(appliance)
            share = registry.get_watt_share(appliance) if col else 0.0
            envelope = appliance_data[col].values * share if col else np.zeros(168)
            self.deferrable_envelope.append(envelope.astype(np.float32))
            self.deferrable_kwh_per_hour.append(float(appliance["watts"]) / 1000.0)

        if self.deferrable_envelope:
            self.deferrable_loads = np.column_stack(self.deferrable_envelope).astype(np.float32)
        else:
            self.deferrable_loads = np.zeros((168, 0), dtype=np.float32)
        self.deferrable_kwh_per_hour = np.asarray(self.deferrable_kwh_per_hour, dtype=np.float32)
        self.n_deferrable = registry.n_deferrable

    def _reset_state(self):
        self.current_hour     = 0
        # Start at min_soc + a margin so episodes don't begin already in penalty
        self.battery_level    = max(0.5, self.min_soc + 0.1) * self.battery_capacity
        self.deferred_skips   = np.zeros(self.n_deferrable, dtype=np.float32)  # FIX #3
        self.hours_since_run  = np.zeros(self.n_deferrable, dtype=np.float32)
        self.daily_runtime    = np.zeros(self.n_deferrable, dtype=np.float32)
        self.weekly_runtime   = np.zeros(self.n_deferrable, dtype=np.float32)  # NEW
        self.last_discharge_rate = 0.0
        self.last_actual_charge  = 0.0
        self.last_run_now        = np.zeros(self.n_deferrable, dtype=bool)

    def reset(self, seed=None, options=None):
        # FIX #28: honor seed
        super().reset(seed=seed)
        if seed is not None:
            self._np_random = np.random.default_rng(seed)

        # FIX #17: resample scenario each episode if a factory was provided
        if self.episode_factory is not None:
            solar, temp, appl_df, sys_cfg, reg = self.episode_factory()
            self._install_scenario(solar, temp, appl_df, sys_cfg, reg)

        self._reset_state()
        return self._get_obs(), {}

    def _get_obs(self):
        h = self.current_hour
        hour_norm    = (h % 24) / 24
        day_norm     = (h // 24) / 7
        battery_norm = self.battery_level / self.battery_capacity

        # FIX #7: normalize system properties to fixed reference scales,
        # not to themselves. Now policy sees consistent meaning across systems.
        battery_capacity_norm = np.clip(self.battery_capacity / 20.0, 0, 1)
        charge_rate_norm      = np.clip(self.max_charge_rate / 5.0, 0, 1)
        panel_capacity_norm   = np.clip(self.panel_capacity / 10.0, 0, 1)

        # Solar window
        solar_window = np.roll(self.solar_forecast, -h)[:24]
        # Normalize by panel capacity (a fixed-meaning reference) instead of window max
        solar_norm   = np.clip(solar_window / max(self.panel_capacity, 1e-6), 0, 1)

        # Fixed load window
        if h + 24 <= 168:
            fixed_window = self.fixed_loads[h:h+24]
        else:
            fixed_window = np.pad(self.fixed_loads[h:], ((0, h+24-168), (0, 0)))
        fixed_sum  = fixed_window.sum(axis=1)
        # Normalize against a fixed reference (5 kWh/h is a generous household max)
        fixed_norm = np.clip(fixed_sum / 5.0, 0, 1)

        # Deferrable load windows — pad to MAX_DEFERRABLE
        def_windows = []
        for i in range(MAX_DEFERRABLE):
            if i < self.n_deferrable:
                if h + 24 <= 168:
                    w = self.deferrable_loads[h:h+24, i]
                else:
                    w = np.pad(self.deferrable_loads[h:, i], (0, h+24-168))
                # Normalize by appliance's true hourly draw (a fixed reference)
                ref = max(self.deferrable_kwh_per_hour[i], 1e-6)
                def_windows.append(np.clip(w / ref, 0, 1))
            else:
                def_windows.append(np.zeros(24, dtype=np.float32))

        # FIX #7: pending uses fixed reference (10 kWh), not battery capacity
        pending = np.zeros(MAX_DEFERRABLE, dtype=np.float32)
        # Use skip count / max_hours_without_running as the urgency signal
        for i, appliance in enumerate(self.registry.deferrable):
            limit = self.registry.get_max_hours_without_running(appliance)
            pending[i] = np.clip(self.hours_since_run[i] / max(limit, 1), 0, 1)

        # Temperature window
        if h + 24 <= 168:
            temp_window = self.temp_forecast[h:h+24]
        else:
            temp_window = np.pad(self.temp_forecast[h:], (0, h+24-168))
        temp_norm = np.clip((temp_window - 5) / 35, 0, 1)

        obs = np.concatenate([
            np.asarray([hour_norm, day_norm, battery_norm,
                        battery_capacity_norm, charge_rate_norm, panel_capacity_norm],
                       dtype=np.float32),
            solar_norm.astype(np.float32),
            fixed_norm.astype(np.float32),
            temp_norm.astype(np.float32),
            *(w.astype(np.float32) for w in def_windows),
            pending,
        ])

        return obs

    def step(self, action):
        h = self.current_hour
        action = np.asarray(action, dtype=np.float32)

        # Reset daily runtime at midnight
        if self.current_hour % 24 == 0 and self.current_hour > 0:
            self.daily_runtime = np.zeros(self.n_deferrable, dtype=np.float32)

        # Find AC and heating indices
        ac_action_idx      = next((i for i, a in enumerate(self.registry.deferrable)
                                   if a["name"] == "cooling"), None)
        heating_action_idx = next((i for i, a in enumerate(self.registry.deferrable)
                                   if a["name"] == "heating"), None)

        charge_command = ((action[0] + 1) / 2) * self.max_charge_rate
        current_temp   = self.temp_forecast[h]

        # Threshold: any positive action -> run. Lowering from 0.1 to 0.0
        # removes a small noise band that was making it harder for the policy
        # to learn the action->outcome mapping during early exploration.
        DEADBAND = 0.0
        deferrable_actions = action[1:1 + self.n_deferrable].copy()
        run_now = deferrable_actions > DEADBAND

        # FIX #13: compute mutual-exclusion penalty BEFORE overriding actions.
        # The policy gradient should see the cost of having issued conflicting commands.
        mutual_exclusion_penalty = 0.0
        if ac_action_idx is not None and heating_action_idx is not None:
            ac_on      = run_now[ac_action_idx]
            heating_on = run_now[heating_action_idx]
            if ac_on and heating_on:
                mutual_exclusion_penalty = -5.0
                if current_temp >= 22:
                    run_now[heating_action_idx] = False
                else:
                    run_now[ac_action_idx] = False

        # Enforce max daily runtime
        for i, appliance in enumerate(self.registry.deferrable):
            max_runtime = self.registry.get_max_daily_runtime(appliance)
            if self.daily_runtime[i] >= max_runtime:
                run_now[i] = False

        solar_in   = self.solar_forecast[h]
        fixed_load = float(self.fixed_loads[h].sum()) if self.fixed_loads.size else 0.0

        # FIX #2: Use REAL appliance wattage when run, not the envelope value.
        if self.n_deferrable > 0:
            deferrable_actual = np.where(run_now,
                                         self.deferrable_kwh_per_hour,
                                         0.0)
            # Track skips: an hour where the predictor said the appliance "wants" to
            # run but the agent didn't run it.
            wants_to_run     = self.deferrable_loads[h] > 0
            newly_skipped    = (wants_to_run & ~run_now).astype(np.float32)
        else:
            deferrable_actual = np.zeros(0, dtype=np.float32)
            newly_skipped     = np.zeros(0, dtype=np.float32)

        total_deferrable = float(deferrable_actual.sum())
        total_load = fixed_load + total_deferrable

        # FIX #3: track skip count instead of energy bucket
        self.deferred_skips += newly_skipped
        # Reset skip counter when we run
        for i in range(self.n_deferrable):
            if run_now[i]:
                self.deferred_skips[i] = 0

        # FIX #12: enforce min_soc as a HARD discharge limit
        usable_energy = max(0.0, self.battery_level - self.min_soc * self.battery_capacity)
        max_discharge_now = min(self.max_discharge_rate, usable_energy)

        # Energy balance
        net = solar_in - total_load - charge_command

        if net >= 0:
            actual_charge  = min(net, self.max_charge_rate,
                                 self.battery_capacity - self.battery_level)
            discharge_rate = 0.0
        else:
            discharge_rate = min(-net, max_discharge_now)
            actual_charge  = 0.0

        self.last_discharge_rate = float(discharge_rate)
        self.last_actual_charge  = float(actual_charge)
        # Expose what was ACTUALLY run this hour (for inference/display).
        # Differs from action because env clamps for daily_runtime cap and
        # AC/heater mutual exclusion.
        self.last_run_now = run_now.copy() if self.n_deferrable > 0 else np.array([], dtype=bool)

        unmet_demand = float(max(0.0, -net - discharge_rate))

        prev_battery       = self.battery_level
        self.battery_level = float(np.clip(
            self.battery_level + actual_charge - discharge_rate,
            0.0, self.battery_capacity
        ))
        battery_pct = self.battery_level / self.battery_capacity
        delta       = abs(self.battery_level - prev_battery) / self.battery_capacity

        # Update tracking
        for i in range(self.n_deferrable):
            if run_now[i]:
                self.hours_since_run[i]  = 0
                self.daily_runtime[i]   += 1
                self.weekly_runtime[i]  += 1  # accumulates over full episode
            else:
                self.hours_since_run[i] += 1

        # ============== REWARDS (rebalanced v2) ==============
        # Key principle: doing-nothing must be MORE costly than running appliances
        # carefully. The previous version let the agent learn that "off" was safe.

        # Demand satisfaction (still dominant — blackouts are the worst outcome)
        demand_penalty = -unmet_demand * 50.0

        # Battery health: gentle. The hard min_soc constraint already prevents damage.
        if battery_pct < self.min_soc:
            low_battery_penalty = -5.0
        elif battery_pct < self.min_soc + 0.10:
            low_battery_penalty = -1.0
        else:
            low_battery_penalty = 0.0

        overcharge_penalty = -2.0 if battery_pct > 0.95 else 0.0  # wasting solar
        smoothness_penalty = 0.0  # removed — was discouraging the agent from acting

        # Deferrable scheduling penalties — STRONGER, scaled by overdue-ness
        # Goal: not running an appliance must hurt MORE per hour than the cost
        # of running it during low-solar periods.
        deferrable_penalty = 0.0
        for i, appliance in enumerate(self.registry.deferrable):
            max_runtime = self.registry.get_max_daily_runtime(appliance)
            limit       = self.registry.get_max_hours_without_running(appliance)
            hours       = self.hours_since_run[i]
            runtime     = self.daily_runtime[i]

            if limit is not None and limit > 0:
                # Linear ramp: starts hurting at 50% of limit, gets severe past limit
                overdue_ratio = hours / limit
                if overdue_ratio > 0.5:
                    deferrable_penalty -= (overdue_ratio - 0.5) * 20.0
                if overdue_ratio > 1.0:
                    deferrable_penalty -= (overdue_ratio - 1.0) * 40.0  # extra slope

            if runtime > max_runtime:
                deferrable_penalty -= 30.0

        # Service bonus: reward EVERY hour an appliance is run, larger when solar covers it.
        # This is the positive signal the agent was missing — currently it only saw
        # penalties for action.
        service_bonus = 0.0
        if self.n_deferrable > 0:
            n_running = int(run_now.sum())
            # Base reward for running anything (encourages action)
            service_bonus += 2.0 * n_running
            # Extra bonus for running when solar is plentiful (encourages timing)
            if solar_in > fixed_load:
                solar_surplus = solar_in - fixed_load
                # Bonus proportional to how much of the surplus we're using
                deferrable_drawn = min(total_deferrable, solar_surplus)
                service_bonus += 5.0 * deferrable_drawn

        # Temperature comfort
        temp_penalty = 0.0
        if ac_action_idx is not None:
            ac_running = run_now[ac_action_idx]
            if current_temp > 30 and not ac_running:
                temp_penalty -= (current_temp - 30) * 0.5
            elif current_temp < 22 and ac_running:
                temp_penalty -= 1.0
        if heating_action_idx is not None:
            heating_running = run_now[heating_action_idx]
            if current_temp < 15 and not heating_running:
                temp_penalty -= (15 - current_temp) * 0.5
            elif current_temp > 16 and heating_running:
                temp_penalty -= 2.0

        reward = float(
            demand_penalty +
            low_battery_penalty + overcharge_penalty +
            smoothness_penalty + deferrable_penalty +
            service_bonus + temp_penalty + mutual_exclusion_penalty
        ) / 10.0

        self.current_hour += 1
        done = self.current_hour >= 7 * 24

        # Terminal reward: at episode end, look at WEEKLY total runtime per appliance
        # vs the expected weekly target (max_daily_runtime × 7). Big bonus for hitting
        # targets, big penalty for missing them. This is the single un-discountable
        # signal that the agent cannot game by procrastinating — it MUST hit the
        # weekly runtime target by end of episode.
        if done and self.n_deferrable > 0:
            terminal_reward = 0.0
            for i, appliance in enumerate(self.registry.deferrable):
                # Weekly target: assume each appliance should run roughly its
                # max_daily_runtime each day, for 7 days.
                max_daily = self.registry.get_max_daily_runtime(appliance)
                # Expected weekly = max_daily × 7, but cap target at a reasonable
                # fraction (don't punish for not running cooling 16h/day all week)
                if appliance["name"] in ("cooling", "heating"):
                    target = max_daily * 7 * 0.5  # AC/heater: half capacity is fine
                else:
                    target = max_daily * 7 * 0.8  # other: 80% of theoretical max
                target = max(target, 1.0)  # at least 1 hour over the week

                actual = self.weekly_runtime[i]
                completion = actual / target

                # Smooth bonus/penalty:
                #   completion = 1.0 -> +50 bonus (perfect)
                #   completion = 0.0 -> -100 penalty (never ran)
                #   completion = 0.5 -> -25 (half-served)
                #   completion > 1.2 -> -20 (over-ran, wasted energy)
                if completion >= 1.2:
                    terminal_reward -= 20.0
                elif completion >= 0.8:
                    terminal_reward += 50.0  # in the sweet spot
                else:
                    # Linear penalty from 0 to -100 as completion goes 0.8 -> 0
                    terminal_reward -= (0.8 - completion) * 125.0

            reward += terminal_reward / 10.0

        return self._get_obs(), reward, done, False, {}

In [55]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pandas as pd
import numpy as np
import random

df = pd.read_parquet("final_data.parquet")

# FIX #6 + Option A: appliance pools use WATTAGE RANGES, not fixed values.
# At every episode we sample a fresh wattage uniformly from the range, so the
# policy sees continuous wattage variation and generalizes to whatever value
# the user types in at inference time.
#
# Each entry: (name, min_watts, max_watts).
# Ranges are wide enough to cover realistic Lebanese household appliances.
DEFERRABLE_POOL = [
    {"name": "washer",       "watts_range": (500,  2000)},
    {"name": "water_heater", "watts_range": (1000, 3500)},
    {"name": "range_oven",   "watts_range": (800,  3000)},
    {"name": "cooling",      "watts_range": (500,  2500)},
    {"name": "heating",      "watts_range": (800,  3000)},
]

# Fixed appliances also get wattage ranges so users with different-sized
# fridges/TVs/lights see a policy that's robust to those variations too.
FIXED_POOL = [
    {"name": "refrigerator", "watts_range": (100,  500)},
    {"name": "freezer",      "watts_range": (80,   400)},
    {"name": "lighting",     "watts_range": (100,  600)},
    {"name": "tv",           "watts_range": (80,   500)},
    {"name": "ceiling_fan",  "watts_range": (40,   120)},
]

# FIX #28: seedable scenario sampler
_GLOBAL_RNG = random.Random(42)
np.random.seed(42)


def sample_appliance(template, rng, deferrable):
    """Sample a fresh wattage from the template's range."""
    lo, hi = template["watts_range"]
    return {
        "name":       template["name"],
        "watts":      rng.uniform(lo, hi),
        "deferrable": deferrable,
    }


def sample_scenario(rng=None):
    """Returns (solar, temp_forecast, appliance_data, system_config, registry).
    Used both at env creation and on every reset (FIX #17)."""
    rng = rng or _GLOBAL_RNG

    system_config = {
        "battery_capacity_kwh":  rng.choice([2.4, 4.8, 7.2]),
        "max_charge_rate_kw":    rng.uniform(1.0, 3.0),
        "max_discharge_rate_kw": rng.uniform(1.0, 3.0),
        "min_soc_pct":           0.30,
        "panel_capacity_kw":     rng.uniform(0.8, 3.0),
    }

    # Sample a varying NUMBER of deferrable appliances (1 to all 5 categories).
    # Each gets a freshly-sampled wattage from its range.
    n_def = rng.randint(1, len(DEFERRABLE_POOL))
    deferrable_templates = rng.sample(DEFERRABLE_POOL, n_def)
    deferrable = [sample_appliance(t, rng, deferrable=True)
                  for t in deferrable_templates]

    # Sample a varying number of fixed appliances too (at least the always-on
    # essentials: fridge + lighting). Other fixed appliances are present
    # with probability 0.7 each.
    fixed = []
    for t in FIXED_POOL:
        if t["name"] in ("refrigerator", "lighting") or rng.random() < 0.7:
            fixed.append(sample_appliance(t, rng, deferrable=False))

    all_appliances = fixed + deferrable
    registry = ApplianceRegistry(all_appliances)

    bldg_id = rng.choice(list(df["building_id"].unique()))
    bldg_df = df[df["building_id"] == bldg_id].reset_index(drop=True)
    start = rng.randint(0, len(bldg_df) - 168)
    df_week = bldg_df.iloc[start:start + 168].reset_index(drop=True)

    solar = get_solar_from_data(df_week,
                                panel_capacity_kw=system_config["panel_capacity_kw"],
                                latitude=36.7, longitude=-119.7)
    appliance_data = df_week[APPLIANCE_COLS]
    temp_forecast  = df_week["weather_drybulb_temp_c"].values.astype(np.float32)

    return solar, temp_forecast, appliance_data, system_config, registry


def make_env(env_seed=None):
    """FIX #17: pass an episode_factory that resamples on every reset."""
    def _init():
        rng = random.Random(env_seed) if env_seed is not None else _GLOBAL_RNG

        def factory():
            return sample_scenario(rng)

        solar, temp, appl_df, sys_cfg, registry = factory()
        env = SolarSchedulingEnv(solar, temp, appl_df, sys_cfg, registry,
                                 episode_factory=factory)
        return env
    return _init


# Sanity check on a single env
check_env(make_env(env_seed=0)())

# 16 parallel envs, each with its own seed for diversity
vec_env = DummyVecEnv([make_env(env_seed=i) for i in range(16)])
vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True, clip_reward=10.0)

# FIX #14, #15: better PPO hyperparameters
# - higher LR (1e-5 was too low for the reward scale)
# - lower entropy (0.05 over-explores in this continuous setting)
# - more epochs per rollout to get more from each batch
model = PPO(
    "MlpPolicy", vec_env,
    learning_rate = 1e-4,
    n_steps       = 168,
    batch_size    = 256,
    n_epochs      = 4,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.1,
    # Higher entropy keeps exploration alive longer. The big terminal reward
    # makes gradients larger; we need strong exploration so the agent doesn't
    # commit too early to "off" before experiencing the terminal bonus.
    ent_coef      = 0.05,
    vf_coef       = 0.5,
    max_grad_norm = 0.5,
    target_kl     = 0.03,
    policy_kwargs = dict(
        net_arch     = [256, 256, 256],
        # Higher initial std (≈1.65) ensures the agent samples a wide range of
        # actions in early training, including positive ("on") values.
        log_std_init = 0.5,
        ortho_init   = True,
    ),
    seed    = 42,
    verbose = 1,
)

# CRITICAL: bias the action distribution to START leaning toward "on".
# Without this, the policy starts at action_mean ≈ 0, and with deadband=0
# that means ~50% of early samples are "off" — but with ent_coef encouraging
# exploration, the agent can quickly collapse to a low-action regime if
# turning off ever looks rewarding short-term. We bias the final layer's bias
# vector so the initial action means are around +0.5 (mostly "on").
# The policy is free to learn negative actions as needed; this just sets
# the starting point on the right side of the local minimum.
import torch as _torch
with _torch.no_grad():
    bias = model.policy.action_net.bias
    # Strong positive bias: +1.0 means initial actions are pinned near +1
    # (definitely "on"). Combined with high std, the agent samples a wide
    # range but anchored on the "on" side.
    bias.data.fill_(1.0)
    print(f"Initialized action bias to +1.0 (was zero). Shape: {bias.shape}")

model.learn(total_timesteps=1_000_000)
vec_env.save("vec_normalize.pkl")
model.save("solar_ppo")

Using cpu device
Initialized action bias to +1.0 (was zero). Shape: torch.Size([13])
-----------------------------
| time/              |      |
|    fps             | 1250 |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2688 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 1287         |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 5376         |
| train/                  |              |
|    approx_kl            | 0.0016865167 |
|    clip_fraction        | 0.0515       |
|    clip_range           | 0.1          |
|    entropy_loss         | -25          |
|    explained_variance   | -0.0725      |
|    learning_rate        | 0.0001       |
|    loss                 | -1.13        |
|    n_updates            | 4            |
|    policy_gradient_loss | -0.00352     |
|    std         

In [15]:
from stable_baselines3 import PPO

user_system = {
    "battery_capacity_kwh":  7.2,
    "max_charge_rate_kw":    2.0,
    "max_discharge_rate_kw": 2.0,
    "min_soc_pct":           0.30,
    "panel_capacity_kw":     3.2,
    "panel_tilt_deg":        34,
    "panel_azimuth_deg":     180,
}

import xgboost as xgb

def predict_appliance_usage(weather_df, model, sqft, bedrooms, hvac_type):
    """FIX #9: handle both single-output and multi-output XGBRegressor models."""
    features = weather_df.copy()
    features["sqft"] = sqft
    features["bedrooms"] = bedrooms
    features["hvac_cooling_type"] = hvac_type

    features = pd.get_dummies(features, columns=["bedrooms", "hvac_cooling_type"])

    expected = model.get_booster().feature_names
    if expected is not None:
        for col in expected:
            if col not in features.columns:
                features[col] = 0
        features = features[expected]

    for col in features.columns:
        features[col] = pd.to_numeric(features[col], errors="coerce").fillna(0)

    features = features.astype("float32")

    pred = model.predict(features).clip(min=0)
    pred = np.asarray(pred)

    # Handle both single-output (1D) and multi-output (2D) cases
    if pred.ndim == 1:
        # Single-output regressor: replicate prediction across all appliance cols
        # (this is a graceful fallback — ideally model is multi-output)
        print("Warning: XGBoost is single-output. Using as total load split equally — "
              "consider retraining as multi-output regressor.")
        per_col = pred.reshape(-1, 1) / len(APPLIANCE_COLS)
        pred_2d = np.tile(per_col, (1, len(APPLIANCE_COLS)))
        return pd.DataFrame(pred_2d, columns=APPLIANCE_COLS, index=weather_df.index)

    if pred.shape[1] != len(APPLIANCE_COLS):
        raise ValueError(
            f"XGBoost output has {pred.shape[1]} columns but APPLIANCE_COLS has "
            f"{len(APPLIANCE_COLS)}. Retrain model with the matching target order."
        )
    return pd.DataFrame(pred, columns=APPLIANCE_COLS, index=weather_df.index)


# FIX #26, #27: drop the unused VecNormalize wrapper at inference. norm_obs=False
# during training means raw obs were fed to the policy — so we just load the policy.
ppo_model = PPO.load("solar_ppo")


def generate_weekly_schedule(latitude, longitude, start_date,
                              system_config, appliance_list,
                              ups_csv_path=None,
                              household_profile=None):

    registry = ApplianceRegistry(appliance_list)
    month    = pd.Timestamp(start_date).month

    solar, temp_forecast, weather_df = get_solar_and_features_from_forecast(
        latitude          = latitude,
        longitude         = longitude,
        start_date        = start_date,
        panel_capacity_kw = system_config["panel_capacity_kw"],
        tilt              = system_config.get("panel_tilt_deg", None),
        azimuth           = system_config.get("panel_azimuth_deg", 180),
    )

    # Feasibility check: estimate weekly energy demand vs solar generation.
    # Helps the user understand if their system can support their appliance load.
    fixed_daily_kwh = sum(a["watts"] / 1000 * 16 for a in registry.fixed)  # ~16h avg
    deferrable_daily_kwh = 0
    for a in registry.deferrable:
        max_run = CATEGORY_CONSTRAINTS.get(a["name"], {}).get("max_daily_runtime", 4)
        # Use realistic average runtime (60% of max for most, less for AC/heat)
        avg_run = max_run * (0.5 if a["name"] in ("cooling", "heating") else 0.8)
        deferrable_daily_kwh += a["watts"] / 1000 * avg_run
    weekly_demand = (fixed_daily_kwh + deferrable_daily_kwh) * 7
    weekly_solar  = float(solar.sum())
    print(f"Estimated weekly demand: {weekly_demand:.1f} kWh")
    print(f"Forecast weekly solar:   {weekly_solar:.1f} kWh")
    if weekly_demand > weekly_solar * 1.3:
        print(f"WARNING: demand exceeds solar generation by "
              f"{(weekly_demand/weekly_solar - 1)*100:.0f}%. "
              f"Some appliances will likely not run as scheduled.")

    if ups_csv_path is not None:
        if not seq2point_models:
            raise RuntimeError("NILM models not loaded — cannot use UPS path.")
        print("Using NILM disaggregation from UPS history (as naive next-week forecast)...")
        appliance_df = disaggregate_seq2point(
            ups_csv_path, seq2point_models, nilm_scaler
        )
        appliance_df = scale_nilm_by_weather(appliance_df, weather_df, month)

    elif household_profile is not None:
        print("Using XGBoost with household profile...")
        features, sqft, bedrooms, hvac_type = profile_to_xgb_features(
            household_profile, weather_df
        )
        xgb_model = xgb.XGBRegressor()
        xgb_model.load_model("usage_prediction.json")
        appliance_df = predict_appliance_usage(
            weather_df, xgb_model, sqft, bedrooms, hvac_type
        )
        appliance_df = scale_predictions_by_profile(
            appliance_df, household_profile, month
        )
    else:
        raise ValueError("Provide either ups_csv_path or household_profile")

    # No episode_factory at inference — we want this exact scenario to be evaluated
    env    = SolarSchedulingEnv(solar, temp_forecast, appliance_df,
                                system_config, registry, episode_factory=None)
    obs, _ = env.reset()

    is_winter = month in [11, 12, 1, 2, 3]
    schedule  = []

    DEADBAND = 0.0  # must match env's deadband

    for hour in range(168):
        action, _ = ppo_model.predict(obs, deterministic=True)
        obs, reward, done, _, _ = env.step(action)

        appliance_row = appliance_df.iloc[hour]

        # FIX #23: use ACTUAL charge from env, not commanded charge
        actual_charge = env.last_actual_charge
        discharge     = env.last_discharge_rate

        if actual_charge > discharge + 1e-6:    battery_state = "Charging"
        elif discharge > actual_charge + 1e-6:  battery_state = "Discharging"
        else:                                   battery_state = "Idle"

        row = {
            "day":           hour // 24 + 1,
            "hour":          f"{hour % 24:02d}:00",
            "solar_kwh":     round(float(solar[hour]), 3),
            "battery_kwh":   round(env.battery_level, 2),
            "battery_state": battery_state,
        }

        for appliance in registry.fixed:
            col = registry.get_category_col(appliance)
            if col:
                category_kwh  = appliance_row.get(col, 0)
                appliance_kwh = float(category_kwh) * registry.get_watt_share(appliance)
                label         = appliance.get("label", appliance["name"])

                if appliance["name"] == "cooling":
                    row[label] = "On" if (not is_winter and appliance_kwh > 0) else "Off"
                elif appliance["name"] == "heating":
                    row[label] = "On" if (is_winter and appliance_kwh > 0) else "Off"
                elif appliance["name"] in ("refrigerator", "freezer"):
                    row[label] = "On"
                elif appliance["name"] == "lighting":
                    row[label] = "On" if appliance_kwh > 0.001 else "Off"
                elif appliance["name"] == "ext_lighting":
                    row[label] = "On" if appliance_kwh > 0.005 else "Off"
                elif appliance["name"] == "tv":
                    row[label] = "On" if appliance_kwh > 0.01 else "Off"
                else:
                    row[label] = "On" if appliance_kwh > 0 else "Off"

        # Use what the env ACTUALLY ran (after applying daily-runtime caps,
        # mutual exclusion, etc.) — not the raw policy action.
        for i, appliance in enumerate(registry.deferrable):
            label = appliance.get("label", appliance["name"])
            row[label] = "On" if env.last_run_now[i] else "Off"

        row["total_load_kwh"] = round(float(appliance_row.sum()), 3)
        schedule.append(row)

    return pd.DataFrame(schedule)


# --- Run ---
schedule = generate_weekly_schedule(
    latitude=33.8, longitude=35.5,
    start_date="2026-05-02",
    system_config=user_system,
    appliance_list=[
        {"name": "cooling",      "label": "Living room AC", "watts": 900,  "deferrable": True},
        {"name": "tv",           "label": "Living room TV", "watts": 150,  "deferrable": False},
        {"name": "refrigerator", "label": "Refrigerator",   "watts": 100,  "deferrable": False},
        {"name": "freezer",      "label": "Freezer",        "watts": 80,   "deferrable": False},
        {"name": "lighting",     "label": "Lights",         "watts": 200,  "deferrable": False},
        {"name": "range_oven",   "label": "Oven",           "watts": 1500, "deferrable": True},
        {"name": "water_heater", "label": "Water heater",   "watts": 1500, "deferrable": True},
        {"name": "washer",       "label": "Washing machine","watts": 800,  "deferrable": True},
    ],
    household_profile={
        "n_people":              4,
        "home_type":             "apartment",
        "cooling_type":          "split_ac",
        "washer_frequency":      "daily",
        "washer_preferred_time": "morning",
        "hot_water_usage":       "high",
        "cooking_frequency":     "twice_daily",
        "sleep_time":            23,
        "wake_time":             7,
        "home_during_day":       False,
    },
)

schedule.to_csv("weekly_schedule.csv", index=False)
print(schedule)

Estimated weekly demand: 128.9 kWh
Forecast weekly solar:   125.0 kWh
Using XGBoost with household profile...
     day   hour  solar_kwh  battery_kwh battery_state Living room TV  \
0      1  00:00      0.000         2.73   Discharging             On   
1      1  01:00      0.000         2.70   Discharging             On   
2      1  02:00      0.000         2.68   Discharging            Off   
3      1  03:00      0.000         2.67   Discharging            Off   
4      1  04:00      0.000         2.65   Discharging            Off   
..   ...    ...        ...          ...           ...            ...   
163    7  19:00      0.143         7.20          Idle             On   
164    7  20:00      0.013         7.17   Discharging             On   
165    7  21:00      0.000         7.12   Discharging             On   
166    7  22:00      0.000         7.06   Discharging             On   
167    7  23:00      0.000         6.99   Discharging             On   

    Refrigerator Freezer 

In [16]:
from stable_baselines3 import PPO
model = PPO.load("solar_ppo")

In [17]:
# ============================================================
# Evaluation: PPO vs solar-aware heuristic baseline
# ============================================================
import numpy as np
import random as _random
import pandas as pd

N_EPISODES = 50
SEED       = 1234

# Self-contained scenario sampler (mirrors the one in the training cell so the
# evaluation works even if training wasn't re-run in this kernel session).
_eval_df = pd.read_parquet("final_data.parquet")

DEFERRABLE_POOL_EVAL = [
    {"name": "washer",       "watts_range": (500,  2000)},
    {"name": "water_heater", "watts_range": (1000, 3500)},
    {"name": "range_oven",   "watts_range": (800,  3000)},
    {"name": "cooling",      "watts_range": (500,  2500)},
    {"name": "heating",      "watts_range": (800,  3000)},
]
FIXED_POOL_EVAL = [
    {"name": "refrigerator", "watts_range": (100,  500)},
    {"name": "freezer",      "watts_range": (80,   400)},
    {"name": "lighting",     "watts_range": (100,  600)},
    {"name": "tv",           "watts_range": (80,   500)},
    {"name": "ceiling_fan",  "watts_range": (40,   120)},
]

def _sample_appliance(template, rng, deferrable):
    lo, hi = template["watts_range"]
    return {"name": template["name"],
            "watts": rng.uniform(lo, hi),
            "deferrable": deferrable}

def sample_eval_scenario(rng):
    system_config = {
        "battery_capacity_kwh":  rng.choice([2.4, 4.8, 7.2]),
        "max_charge_rate_kw":    rng.uniform(1.0, 3.0),
        "max_discharge_rate_kw": rng.uniform(1.0, 3.0),
        "min_soc_pct":           0.30,
        "panel_capacity_kw":     rng.uniform(0.8, 3.0),
    }
    n_def = rng.randint(1, len(DEFERRABLE_POOL_EVAL))
    deferrable = [_sample_appliance(t, rng, True)
                  for t in rng.sample(DEFERRABLE_POOL_EVAL, n_def)]
    fixed = []
    for t in FIXED_POOL_EVAL:
        if t["name"] in ("refrigerator", "lighting") or rng.random() < 0.7:
            fixed.append(_sample_appliance(t, rng, False))

    registry = ApplianceRegistry(fixed + deferrable)

    bldg_id = rng.choice(list(_eval_df["building_id"].unique()))
    bldg_df = _eval_df[_eval_df["building_id"] == bldg_id].reset_index(drop=True)
    start = rng.randint(0, len(bldg_df) - 168)
    df_week = bldg_df.iloc[start:start + 168].reset_index(drop=True)

    solar = get_solar_from_data(df_week,
                                panel_capacity_kw=system_config["panel_capacity_kw"],
                                latitude=36.7, longitude=-119.7)
    appliance_data = df_week[APPLIANCE_COLS]
    temp_forecast  = df_week["weather_drybulb_temp_c"].values.astype(np.float32)

    return solar, temp_forecast, appliance_data, system_config, registry


def evaluate_policy(policy_fn, n_episodes, seed):
    rng = _random.Random(seed)
    np.random.seed(seed)

    metrics = {
        'self_consumption':      [],
        'demand_satisfaction':   [],
        'soc_violation_rate':    [],
        'solar_alignment':       [],
        'deferrable_completion': [],
    }

    for ep in range(n_episodes):
        solar, temp, appliance_data, sys_cfg, registry = sample_eval_scenario(rng)
        env = SolarSchedulingEnv(solar, temp, appliance_data, sys_cfg, registry,
                                 episode_factory=None)
        obs, _ = env.reset()

        solar_generated   = 0.0
        solar_to_load     = 0.0
        solar_to_battery  = 0.0
        total_demand      = 0.0
        unmet_demand      = 0.0
        soc_violations    = 0
        weekly_runtime    = np.zeros(env.n_deferrable)
        solar_per_hour    = np.zeros(168)
        defer_load_hourly = np.zeros(168)

        prev_battery = env.battery_level

        for h in range(168):
            action = policy_fn(env, obs, h)
            obs, _, done, _, _ = env.step(action)

            solar_h = float(env.solar_forecast[h])
            fixed_h = float(env.fixed_loads[h].sum()) if env.fixed_loads.size else 0.0
            defer_h = float((env.deferrable_kwh_per_hour * env.last_run_now).sum()) \
                      if env.n_deferrable > 0 else 0.0
            load_h  = fixed_h + defer_h

            battery_delta = env.battery_level - prev_battery
            charging      = max(0.0,  battery_delta)
            discharging   = max(0.0, -battery_delta)

            s_to_load = min(solar_h, load_h)
            s_to_batt = min(max(0.0, solar_h - s_to_load), charging)
            solar_generated  += solar_h
            solar_to_load    += s_to_load
            solar_to_battery += s_to_batt

            served_h = s_to_load + discharging
            total_demand += load_h
            unmet_demand += max(0.0, load_h - served_h)

            if env.battery_level / env.battery_capacity < env.min_soc:
                soc_violations += 1

            solar_per_hour[h]    = solar_h
            defer_load_hourly[h] = defer_h
            if env.n_deferrable > 0:
                weekly_runtime += env.last_run_now.astype(float)

            prev_battery = env.battery_level
            if done:
                break

        metrics['self_consumption'].append(
            (solar_to_load + solar_to_battery) / max(solar_generated, 1e-6)
        )
        metrics['demand_satisfaction'].append(
            1.0 - (unmet_demand / max(total_demand, 1e-6))
        )
        metrics['soc_violation_rate'].append(soc_violations / 168.0)

        if solar_per_hour.sum() > 0 and defer_load_hourly.sum() > 0:
            alignment = float(
                (solar_per_hour * defer_load_hourly).sum() /
                (np.linalg.norm(solar_per_hour) * np.linalg.norm(defer_load_hourly))
            )
        else:
            alignment = 0.0
        metrics['solar_alignment'].append(alignment)

        if env.n_deferrable > 0:
            completions = []
            for i, a in enumerate(env.registry.deferrable):
                max_daily = env.registry.get_max_daily_runtime(a)
                target    = max_daily * 7 * (0.5 if a["name"] in ("cooling","heating") else 0.8)
                target    = max(target, 1.0)
                completions.append(min(weekly_runtime[i] / target, 1.5))
            metrics['deferrable_completion'].append(float(np.mean(completions)))
        else:
            metrics['deferrable_completion'].append(1.0)

    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in metrics.items()}


# ---- Policy implementations ----

def heuristic_policy(env, obs, hour):
    avg_solar = env.solar_forecast.mean()
    run_them  = env.solar_forecast[hour] > avg_solar
    action    = np.full(1 + MAX_DEFERRABLE, -1.0, dtype=np.float32)
    action[0] = 1.0
    if run_them:
        action[1:1 + env.n_deferrable] = 1.0
    return action

def ppo_policy(env, obs, hour):
    action, _ = model.predict(obs, deterministic=True)
    return np.asarray(action, dtype=np.float32)


# ---- Run benchmark ----
print(f"Evaluating on {N_EPISODES} scenarios (seed={SEED})...\n")

print("Running heuristic baseline...")
heur_results = evaluate_policy(heuristic_policy, N_EPISODES, SEED)

print("Running PPO model...")
ppo_results  = evaluate_policy(ppo_policy,       N_EPISODES, SEED)

# ---- Print comparison ----
print(f"\n{'Metric':<28} {'Heuristic':>18} {'PPO':>18} {'Δ':>10}")
print("-" * 76)
for key in ['self_consumption', 'demand_satisfaction', 'soc_violation_rate',
            'solar_alignment', 'deferrable_completion']:
    h_mean, h_std = heur_results[key]
    p_mean, p_std = ppo_results[key]
    delta = p_mean - h_mean
    if key == 'soc_violation_rate':
        arrow = "↓" if delta < -0.001 else ("↑" if delta > 0.001 else " ")
    else:
        arrow = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else " ")
    print(f"{key:<28} {h_mean:>7.3f} ± {h_std:.3f}   "
          f"{p_mean:>7.3f} ± {p_std:.3f}  {arrow}{abs(delta):>6.3f}")
print("-" * 76)
print("Notes:")
print("  self_consumption    : higher = less wasted solar")
print("  demand_satisfaction : higher = more loads served")
print("  soc_violation_rate  : LOWER  = battery stays above min_soc more often")
print("  solar_alignment     : higher = appliances run during solar peaks")
print("  deferrable_completion: higher (capped at 1.5) = appliances meet their targets")

Evaluating on 50 scenarios (seed=1234)...

Running heuristic baseline...
Running PPO model...

Metric                                Heuristic                PPO          Δ
----------------------------------------------------------------------------
self_consumption               0.793 ± 0.265     0.661 ± 0.293  ↓ 0.132
demand_satisfaction            0.221 ± 0.150     0.590 ± 0.320  ↑ 0.369
soc_violation_rate             0.000 ± 0.000     0.000 ± 0.000    0.000
solar_alignment                0.644 ± 0.203     0.135 ± 0.157  ↓ 0.510
deferrable_completion          1.169 ± 0.175     0.324 ± 0.241  ↓ 0.845
----------------------------------------------------------------------------
Notes:
  self_consumption    : higher = less wasted solar
  demand_satisfaction : higher = more loads served
  soc_violation_rate  : LOWER  = battery stays above min_soc more often
  solar_alignment     : higher = appliances run during solar peaks
  deferrable_completion: higher (capped at 1.5) = appliances meet